Introduction to SQL
Introduction to SQL
Welcome to your first taste of SQL (Structured Query Language). From Week 6 onward, the datasets you need for lessons, tasks, assignments, and assessments no longer ship as CSV files in these folders — they live in the cohort's cloud database, and you will pull them out yourself with SQL.

This is deliberate. Reading a CSV is a one-line trick. Knowing how to ask a database for exactly the data you need is a core data-science skill that shows up in every real job. You'll learn it a little at a time, right alongside pandas and machine learning.

1. What is a database, really?
A relational database is an organized collection of tables. A table is just rows and columns — like a spreadsheet or a pandas DataFrame — but stored on a server, queryable by many people at once, and grouped into schemas (folders for tables).

In our cohort database (assessmentDB), every week has its own schema:

Schema	Week
week06	Numpy & Pandas-1
week07	Data Analysis (Pandas-2)
week08	Advanced Data Analysis (Pandas-1)
week09	Advanced Data Analysis (Pandas-2)
week14–week19	Regression, Classification, Clustering & NLP
A full table name is schema.table, e.g. week08.wb_gdp_2013.

2. The one command you'll use most: SELECT
SELECT *
FROM week08.wb_gdp_2013;
SELECT * — give me all columns.
FROM week08.wb_gdp_2013 — from this table.
Pick specific columns instead of *:

SELECT country, year, [NY.GDP.MKTP.CD]
FROM week08.wb_gdp_2013;
Bracket rule: if a column name has a space or a dot (like Max TemperatureC or NY.GDP.MKTP.CD), wrap it in square brackets: [Max TemperatureC]. Our real-world datasets have plenty of these.

Filter rows with WHERE
SELECT country, [NY.GDP.MKTP.CD]
FROM week08.wb_gdp_2013
WHERE [NY.GDP.MKTP.CD] > 1000000000000;   -- GDP over 1 trillion
Common operators: =, <> (not equal), >, <, >=, <=, AND, OR, LIKE 'Nigeria%' (pattern match), IN ('Ghana','Kenya'), IS NULL.

Sort with ORDER BY and cap rows with TOP
SELECT TOP 10 country, [NY.GDP.MKTP.CD]
FROM week08.wb_gdp_2013
ORDER BY [NY.GDP.MKTP.CD] DESC;   -- 10 biggest economies
Note: SQL Server uses SELECT TOP 10 ..., not LIMIT 10 (that's MySQL/Postgres).

3. Summarising data: GROUP BY + aggregates
Aggregate functions collapse many rows into one number: COUNT(), SUM(), AVG(), MIN(), MAX().

-- How many countries are in the table?
SELECT COUNT(*) AS n_countries
FROM week08.wb_pop_2013;
GROUP BY computes an aggregate per group:

-- Average max temperature per month in London, 2014
SELECT LEFT(GMT, 7) AS month, AVG([Max TemperatureC]) AS avg_max_temp
FROM week06.london_2014
GROUP BY LEFT(GMT, 7)
ORDER BY month;
4. Combining tables: JOIN
Often the data you want is split across tables. A JOIN stitches them together on a shared key. Here GDP and population live in two tables, both keyed by country:

SELECT g.country,
       g.[NY.GDP.MKTP.CD] AS gdp,
       p.[SP.POP.TOTL]    AS population
FROM week08.wb_gdp_2013 AS g
JOIN week08.wb_pop_2013 AS p
  ON g.country = p.country;
AS g / AS p are short aliases so you don't retype the full table name.
ON g.country = p.country is the matching rule.
5. Connecting from Python
You'll run your SQL from notebooks using a tiny helper we provide, db/dataraflow_db.py. Paste this bootstrap near the top of any notebook — it finds the project root no matter how deep the notebook is, then imports run_query:

# --- connect to the cohort database (read-only) ---
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query
Now every query returns a pandas DataFrame — carry on exactly as you would after pd.read_csv:

df = run_query("SELECT * FROM week08.wb_gdp_2013")
df.head()

# You can do the heavy lifting in SQL and the analysis in pandas:
top = run_query("""
    SELECT TOP 10 country, [NY.GDP.MKTP.CD] AS gdp
    FROM week08.wb_gdp_2013
    ORDER BY [NY.GDP.MKTP.CD] DESC
""")

# Save a result to CSV whenever you need a local copy for modelling:
top.to_csv("top10_gdp.csv", index=False)
6. Getting access (do this once)
You connect with read-only credentials — you can SELECT data but never change it, so you can experiment freely without breaking anything for others.

Get your IP whitelisted. The database firewall blocks unknown computers. Find your public IP (search "what is my IP" in a browser) and send it to your program admin to be added to the allow-list. If your internet connection changes, your IP may change and you'll need to re-submit it.
Add the credentials to a local .env file in the project root (ask the admin for the read-only username and password): SQL_SERVER_NAME=df-cohort2.database.windows.net SQL_SERVER_DATABASE=assessmentDB SQL_READONLY_USERNAME=intern_readonly SQL_READONLY_PASSWORD=<given to you by the admin>
Install the driver once: pip install pymssql pandas sqlalchemy python-dotenv
Test it:

from db.dataraflow_db import run_query
print(run_query("SELECT COUNT(*) AS n FROM week08.wb_gdp_2013"))
If you see a row count, you're connected. 🎉

7. Where to go each week
From Week 6 on, every data-bearing week has a Data Access (SQL).md file listing that week's schema, its tables, and worked examples of pulling, joining, aggregating, and exporting the exact data the week's tasks need. Start there.

Quick reference
Goal	SQL
All rows/columns	SELECT * FROM week08.wb_gdp_2013
Some columns	SELECT country, year FROM week08.wb_gdp_2013
Filter	... WHERE year = 2013
Sort	... ORDER BY country ASC
First N rows	SELECT TOP 10 ...
Count	SELECT COUNT(*) FROM ...
Group + aggregate	SELECT country, AVG(x) FROM ... GROUP BY country
Join two tables	... FROM a JOIN b ON a.key = b.key
Column with space/dot	[Max TemperatureC], [NY.GDP.MKTP.CD]